Question: Algorithm like RandomForest does not have any parameters just like LogisticRegression has. So what kind of parameters the warm start function used when it retrains on the new data along with previous data?

Answer →
In scikit-learn, the `warm_start=True` parameter allows you to perform **incremental learning**. 

Normally, when you call `.fit()` on a model, it clears any existing state (like trained trees in a Random Forest) and starts training from scratch. With `warm_start=True`, the model **retains its previous state** and adds new components to it upon subsequent `.fit()` calls.

### How it works for Random Forest
In a Random Forest, `warm_start` specifically controls whether to reuse the existing trees:

1.  **First `.fit()`:** Trains the number of trees specified by `n_estimators`.
2.  **Subsequent `.fit()`:** 
    *   If you **don't** increase `n_estimators`, it does nothing (and gives you the warning you saw).
    *   If you **increase** `n_estimators`, it keeps the old trees and only trains the *additional* trees required to reach the new total.

### Why you got that warning
In your code (`from sklearn.py`):
```python
model = RandomForestClassifier(n_estimators=2, warm_start=True)
train(model, X_train, y_train)    # Fits 2 trees
retrain(model, X_train, y_train)  # Tries to fit again, but n_estimators is still 2
```
Since `n_estimators` was still `2` during the second call, the model looked at its state, saw it already had 2 trees, and decided it didn't need to do anything. That's why you saw:
> `UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.`

### How to use it correctly
To actually "add" to the model, you should increment the `n_estimators` attribute before the second fit:

```python
model = RandomForestClassifier(n_estimators=2, warm_start=True)
model.fit(X_train, y_train) # Trains 2 trees

# Now increase the target number of trees
model.n_estimators = 5 
model.fit(X_train, y_train) # Keeps the first 2, and trains 3 NEW trees
```

### When is this useful?
*   **Early Stopping:** You can train a few trees, check the validation accuracy, and decide whether to keep adding more trees or stop.
*   **Memory/Time Management:** Training a massive forest in chunks.
*   **Changing Data:** If you have new data arriving, you can add more trees trained specifically on the new samples (though be careful about bias).

**Note:** For other models like `LogisticRegression`, `warm_start` doesn't add "new parts" but uses the coefficients from the previous fit as the starting point (initialization) for the optimization algorithm, which makes convergence much faster.